In [1]:
import pandas as pd
import numpy as np, json, re
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import make_scorer, mean_squared_error

In [2]:
# --- 1. Load the dataset ---
df_master = pd.read_csv('../../../data/test-final/FINAL_master.csv')
df_jd = pd.read_csv('../../../data/test-final/jds_clean.csv')

In [3]:
# 2. Merge JD embedding into master
df = df_master.merge(
    df_jd[['jd_id','jd_text_embedding']],
    on='jd_id', how='left'
)


In [5]:
# 3. Parsing helpers
def parse_vec(s):
    # generic “[x, y, z]” → np.array
    try:
        return np.array(json.loads(s.replace("'", "'")))
    except:
        parts = re.split(r'[,\s]+', s.strip().lstrip('[').rstrip(']'))
        return np.array([float(x) for x in parts if x])

In [6]:
# 4. Extract vectors
df['struct_vec'] = df['structured_features'].apply(parse_vec)
df['text_vec']   = df['transcript_embedding'].  apply(parse_vec)
df['jd_vec']     = df['jd_text_embedding'].    apply(parse_vec)


In [7]:
# 5. Drop any bad rows
df = df[(df['struct_vec'].map(len)>0)
      & (df['text_vec']  .map(len)>0)
      & (df['jd_vec']    .map(len)>0)]

In [8]:
# 6a. Prepare targets
y_reg = df['Overall_Score'].values
rec_cols = ['rec_Hire','rec_Consider','rec_Reject']
y_cls   = df[rec_cols].values.argmax(axis=1)

In [9]:
# 6b. CV setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)
mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)

In [10]:
# ——— JD‑Only Baseline ———
X_jd = np.vstack(df['jd_vec'].values)

reg_jd = GradientBoostingRegressor(random_state=42)
mse_jd = cross_val_score(reg_jd, X_jd, y_reg, cv=cv, scoring=mse_scorer)
rmse_jd = np.sqrt(-mse_jd)
print("JD‑Only Regression CV RMSE:", np.round(rmse_jd,3))
print("Mean RMSE:", np.round(rmse_jd.mean(),3))

clf_jd = RandomForestClassifier(random_state=42)
acc_jd = cross_val_score(clf_jd, X_jd, y_cls, cv=cv, scoring='accuracy')
print("JD‑Only Classification CV Acc.:", np.round(acc_jd,3))
print("Mean Acc.:", np.round(acc_jd.mean(),3))

JD‑Only Regression CV RMSE: [0.214 0.203 0.23  0.233 0.231]
Mean RMSE: 0.222
JD‑Only Classification CV Acc.: [0.377 0.2   0.3   0.317 0.3  ]
Mean Acc.: 0.299


In [11]:
# ——— Full Fused Baseline ———
X_struct = np.vstack(df['struct_vec'].values)
X_text   = np.vstack(df['text_vec'].  values)
X_fused  = np.hstack([X_struct, X_jd, X_text])

reg_f = GradientBoostingRegressor(random_state=42)
mse_f = cross_val_score(reg_f, X_fused, y_reg, cv=cv, scoring=mse_scorer)
rmse_f = np.sqrt(-mse_f)
print("\nFused (S+JD+T) Regression CV RMSE:", np.round(rmse_f,3))
print("Mean RMSE:", np.round(rmse_f.mean(),3))

clf_f = RandomForestClassifier(random_state=42)
acc_f = cross_val_score(clf_f, X_fused, y_cls, cv=cv, scoring='accuracy')
print("Fused (S+JD+T) Classification CV Acc.:", np.round(acc_f,3))
print("Mean Acc.:", np.round(acc_f.mean(),3))


Fused (S+JD+T) Regression CV RMSE: [0.065 0.073 0.062 0.069 0.07 ]
Mean RMSE: 0.068
Fused (S+JD+T) Classification CV Acc.: [0.967 0.967 0.983 0.983 0.983]
Mean Acc.: 0.977
